# Phishing Model Retraining (Balanced Train Split Version)

This notebook is tuned for your flattened `combined_email_dataset.csv` and for the class imbalance shown in your audit output.

What it does:
- uses your existing combined dataset in Drive
- keeps nearly all rows after safe cleaning
- removes obvious source-specific tokens more aggressively
- balances the **training split only** so the model does not learn an overwhelming safe prior
- keeps validation and test splits untouched so evaluation stays honest


In [4]:
!pip -q install pandas numpy scikit-learn joblib openpyxl pyarrow matplotlib seaborn

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import json
import re
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 180)
sns.set_theme(style='whitegrid')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
PROJECT_ROOT = Path('/content/drive/MyDrive/fyp')
OUTPUT_DIR = PROJECT_ROOT / 'retrained_backend'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATASET_PATH = PROJECT_ROOT / 'combined_email_dataset.csv'

drive_root_npz = sorted(Path('/content/drive/MyDrive').glob('*.npz'))
OPTIONAL_NPZ_PATH = drive_root_npz[0] if len(drive_root_npz) == 1 else None

TEXT_COLUMN = 'text'
LABEL_COLUMN = 'label'
POSSIBLE_SOURCE_COLUMNS = ['source', 'dataset', 'source_dataset', 'origin']

RANDOM_STATE = 42
MAX_FEATURES = 5000
NGRAM_RANGE = (1, 2)
MIN_DF = 2
MAX_DF = 0.95
MIN_TEXT_LENGTH = 15
MIN_PRECISION_FOR_THRESHOLD = 0.90

DROP_DUPLICATES_ON_RAW_TEXT = True
DROP_DUPLICATES_ON_CLEANED_TEXT = False

# Rebalance only the training split. Validation and test stay untouched.
BALANCE_TRAIN_SPLIT = True
MAX_SAFE_TO_PHISHING_RATIO = 3

SOURCE_REGEX_PATTERNS = [
    r'\benron\w*\b',
    r'\bx\s*-?filename\b',
    r'\bx\s*-?origin\b',
    r'\bx\s*-?folder\b',
    r'\bmessage\s*-?id\b',
    r'\bmime\s*-?version\b'
]

HEADER_FIELD_PATTERNS = [
    r'message\s*-?id\s*:', r'date\s*:', r'from\s*:', r'to\s*:', r'cc\s*:', r'bcc\s*:',
    r'subject\s*:', r'mime\s*-?version\s*:', r'content\s*-?type\s*:',
    r'content\s*-?transfer\s*-?encoding\s*:', r'x\s*-?from\s*:', r'x\s*-?to\s*:',
    r'x\s*-?cc\s*:', r'x\s*-?bcc\s*:', r'x\s*-?folder\s*:', r'x\s*-?origin\s*:',
    r'x\s*-?filename\s*:', r'return\s*-?path\s*:', r'reply\s*-?to\s*:', r'sender\s*:'
]

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_DATASET_PATH =', RAW_DATASET_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('OPTIONAL_NPZ_PATH =', OPTIONAL_NPZ_PATH)

PROJECT_ROOT = /content/drive/MyDrive/fyp
RAW_DATASET_PATH = /content/drive/MyDrive/fyp/combined_email_dataset.csv
OUTPUT_DIR = /content/drive/MyDrive/fyp/retrained_backend
OPTIONAL_NPZ_PATH = /content/drive/MyDrive/tfidf_matrix.npz


In [7]:
def read_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')

    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    if suffix == '.parquet':
        return pd.read_parquet(path)
    if suffix == '.json':
        return pd.read_json(path)

    raise ValueError(f'Unsupported file format: {suffix}')


def inspect_npz(npz_path):
    if npz_path is None:
        print('No .npz file auto-selected. Skipping .npz inspection.')
        return None

    npz_path = Path(npz_path)
    if not npz_path.exists():
        print(f'.npz file not found at {npz_path}, skipping inspection.')
        return None

    npz_data = np.load(npz_path, allow_pickle=True)
    print('NPZ keys:', list(npz_data.keys()))
    for key in npz_data.files:
        value = npz_data[key]
        print(f'- {key}: shape={getattr(value, "shape", None)}, dtype={getattr(value, "dtype", None)}')
    print('Note: this .npz looks like a sparse feature matrix, not raw text. It is diagnostic only for this notebook.')
    return npz_data


def find_source_column(df):
    for col in POSSIBLE_SOURCE_COLUMNS:
        if col in df.columns:
            return col
    return None


def remove_header_fields_from_flattened_text(text):
    for pattern in HEADER_FIELD_PATTERNS:
        text = re.sub(pattern, ' ', text, flags=re.IGNORECASE)
    return text


def remove_source_patterns(text):
    for pattern in SOURCE_REGEX_PATTERNS:
        text = re.sub(pattern, ' ', text, flags=re.IGNORECASE)
    return text


def clean_email_text(text):
    if not isinstance(text, str):
        return ''

    text = re.sub(r'<.*?>', ' ', text)
    text = text.lower()
    text = remove_header_fields_from_flattened_text(text)
    text = re.sub(r'https?://\S+|www\.\S+', ' URL ', text)
    text = re.sub(r'\b[\w.+-]+@[\w.-]+\.\w+\b', ' EMAIL ', text)
    text = re.sub(r'\b\d+\b', ' NUM ', text)
    text = re.sub(r'(_x000d_|=20|=09)', ' ', text)
    text = remove_source_patterns(text)
    text = re.sub(r'[^a-z0-9_\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def load_training_frame():
    df = read_table(RAW_DATASET_PATH).copy()
    print('Dataset loaded successfully.')
    print('Shape before validation:', df.shape)
    print('Columns:', df.columns.tolist())

    required_columns = [TEXT_COLUMN, LABEL_COLUMN]
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f'Missing required columns: {missing_columns}')

    source_column = find_source_column(df)
    keep_cols = [TEXT_COLUMN, LABEL_COLUMN]
    if source_column:
        keep_cols.append(source_column)

    df = df[keep_cols].copy()
    print('\nRows after keeping needed columns:', len(df))

    df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN]).copy()
    print('Rows after dropna:', len(df))

    df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str)
    df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(int)

    valid_labels = {0, 1}
    actual_labels = set(df[LABEL_COLUMN].unique())
    if not actual_labels.issubset(valid_labels):
        raise ValueError(f'Unexpected label values found: {actual_labels}')

    df = df.rename(columns={TEXT_COLUMN: 'raw_text', LABEL_COLUMN: 'label'})

    if DROP_DUPLICATES_ON_RAW_TEXT:
        before_raw_dedup = len(df)
        df = df.drop_duplicates(subset=['raw_text', 'label']).copy()
        print('Rows after raw-text dedup:', len(df), f'(removed {before_raw_dedup - len(df)})')

    df = df[df['raw_text'].str.strip() != ''].copy()
    print('Rows after removing empty raw text:', len(df))

    df['text'] = df['raw_text'].map(clean_email_text)
    df = df[df['text'].str.strip() != ''].copy()
    print('Rows after cleaning and removing empty cleaned text:', len(df))

    df = df[df['text'].str.len() >= MIN_TEXT_LENGTH].copy()
    print('Rows after minimum length filter:', len(df))

    cleaned_duplicate_count = df.duplicated(subset=['text', 'label']).sum()
    print('Duplicate rows if deduped on cleaned text:', int(cleaned_duplicate_count))

    if DROP_DUPLICATES_ON_CLEANED_TEXT:
        before_clean_dedup = len(df)
        df = df.drop_duplicates(subset=['text', 'label']).copy()
        print('Rows after cleaned-text dedup:', len(df), f'(removed {before_clean_dedup - len(df)})')

    df = df.reset_index(drop=True)
    return df, source_column


def audit_training_frame(df, source_column=None):
    print('\nRows ready for training:', len(df))
    print('\nClass distribution:')
    print(df['label'].value_counts().rename(index={0: 'safe', 1: 'phishing'}))

    text_lengths = df['text'].str.len()
    print('\nAverage cleaned text length:', round(text_lengths.mean(), 2))
    print('Median cleaned text length:', round(text_lengths.median(), 2))

    suspicious_tokens = ['enron', 'received', 'content_type']
    print('\nToken counts in cleaned text:')
    token_counts = {}
    for token in suspicious_tokens:
        count = int(df['text'].str.contains(token, regex=False).sum())
        token_counts[token] = count
        print(f'- {token}: {count}')

    if source_column and source_column in df.columns:
        print(f'\nDetected source column: {source_column}')
        source_table = pd.crosstab(df[source_column], df['label'])
        display(source_table)

    print('\nSample cleaned rows:')
    display(df[['raw_text', 'text', 'label']].sample(min(len(df), 5), random_state=RANDOM_STATE))

    for token, count in token_counts.items():
        if count > 0 and token != 'received':
            print(f'\nExamples still containing token: {token}')
            display(df[df['text'].str.contains(token, regex=False)][['text', 'label']].head(3))


def rebalance_training_split(train_df):
    if not BALANCE_TRAIN_SPLIT:
        return train_df.copy()

    phishing_df = train_df[train_df['label'] == 1].copy()
    safe_df = train_df[train_df['label'] == 0].copy()

    max_safe = min(len(safe_df), len(phishing_df) * MAX_SAFE_TO_PHISHING_RATIO)
    safe_sample = safe_df.sample(n=max_safe, random_state=RANDOM_STATE)

    balanced_df = pd.concat([safe_sample, phishing_df], ignore_index=True)
    balanced_df = balanced_df.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

    print('\nTraining split before rebalance:')
    print(train_df['label'].value_counts().rename(index={0: 'safe', 1: 'phishing'}))
    print('\nTraining split after rebalance:')
    print(balanced_df['label'].value_counts().rename(index={0: 'safe', 1: 'phishing'}))

    return balanced_df

In [8]:
npz_data = inspect_npz(OPTIONAL_NPZ_PATH)
df, source_column = load_training_frame()
audit_training_frame(df, source_column)

NPZ keys: ['indices', 'indptr', 'format', 'shape', 'data']
- indices: shape=(76398482,), dtype=int32
- indptr: shape=(534908,), dtype=int32
- format: shape=(), dtype=|S3
- shape: shape=(2,), dtype=int64
- data: shape=(76398482,), dtype=float64
Note: this .npz looks like a sparse feature matrix, not raw text. It is diagnostic only for this notebook.
Dataset loaded successfully.
Shape before validation: (534907, 2)
Columns: ['text', 'label']

Rows after keeping needed columns: 534907
Rows after dropna: 534906
Rows after raw-text dedup: 534906 (removed 0)
Rows after removing empty raw text: 534906
Rows after cleaning and removing empty cleaned text: 534903
Rows after minimum length filter: 534887
Duplicate rows if deduped on cleaned text: 10901

Rows ready for training: 534887

Class distribution:
label
safe        528360
phishing      6527
Name: count, dtype: int64

Average cleaned text length: 1716.27
Median cleaned text length: 845.0

Token counts in cleaned text:
- enron: 3734
- recei

,raw_text,text,label
409922,"Message-ID: Date: Thu, 1 Jun 2000 10:13:00 -0700 (PDT) From: tana.jones@enron.com To: melissa.murphy@enron.com, kim.theriot@enron.com, rhonda.denton@enron.com, jefferson.sorens...",thu jun pdt financial trading agreements database link text plain charset us ascii 7bit b x tana jones x melissa ann murphy kim s theriot rhonda l denton jefferson d sorenson l...,0
436019,"Message-ID: Date: Mon, 15 Oct 2001 10:42:34 -0700 (PDT) From: andy.pace@enron.com To: daniel.muschar@enron.com, steve.pan@enron.com, jason.kaniss@enron.com, j..broderick@enron....",mon oct pdt results of football pool text plain charset us ascii 7bit x pace andy x muschar daniel pan steve kaniss jason broderick paul j broussard richard schneider bryce ste...,0
434587,"Message-ID: Date: Fri, 2 Nov 2001 07:45:28 -0800 (PST) From: kay.quigley@enron.com To: dutch.quigley@enron.com Subject: Re: Good Morning Mime-Version: 1.0 Content-Type: text/pl...",fri nov pst re good morning text plain charset us ascii 7bit x quigley kay x quigley dutch x x b exmerge quigley dutch private folders kq dq quigley d dutch quigley pst i did g...,0
279897,"Message-ID: Date: Fri, 9 Feb 2001 01:22:00 -0800 (PST) From: darron.giron@enron.com To: carole.frank@enron.com Subject: Credit Report--2/9/01 Mime-Version: 1.0 Content-Type: te...",fri feb pst credit report text plain charset us ascii 7bit x darron c giron x carole frank x x b darron_giron_jun2001 notes folders all documents giron d dgiron nsf forwarded b...,0
309889,"Message-ID: Date: Thu, 19 Jul 2001 15:58:21 -0700 (PDT) From: legal To: marie.heard@enron.com Subject: FW: Canadian Hunter Resources ISDA Mime-Version: 1.0 Content-Type: text/p...",thu jul pdt legal fw canadian hunter resources isda text plain charset us ascii 7bit x taylor mark e legal x heard marie x x b mtaylo1 non privileged taylor mark e legal sent i...,0



Examples still containing token: enron


,text,label
60,mon feb pst fortune most admired ranking text plain charset us ascii 7bit x office of the chairman x all_enron_north america ec employees all communications mass mailing list e...,0
440,mon feb pst fortune most admired ranking text plain charset us ascii 7bit x office of the chairman x all_enron_north america ec employees all communications mass mailing list e...,0
616,mon feb pst fortune most admired ranking text plain charset us ascii 7bit x office of the chairman x all_enron_north america ec employees all communications mass mailing list e...,0



Examples still containing token: content_type


,text,label
521593,been running hammie on all my incoming messages and i noticed that multipart alternative messages are totally hosed they have no content just the mime boundaries for instance t...,0
526843,it s possible i performed the update via rpm u which of course created all the new rulesets as xx_rulename cf rpmnew crud i ll have to start moving things around on thu sep mal...,0
529883,on thursday september cet mike burger wrote just loaded up sa from theo s rpms spamassassin and perl mail spamassassin on a rh system with perl running on it i m getting messag...,0


In [9]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df['label'],
    random_state=RANDOM_STATE
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label'],
    random_state=RANDOM_STATE
)

train_fit_df = rebalance_training_split(train_df)

print('\nTrain:', train_df.shape)
print('Validation:', val_df.shape)
print('Test:', test_df.shape)
print('Train used for fitting:', train_fit_df.shape)


Training split before rebalance:
label
safe        422687
phishing      5222
Name: count, dtype: int64

Training split after rebalance:
label
safe        15666
phishing     5222
Name: count, dtype: int64

Train: (427909, 3)
Validation: (53489, 3)
Test: (53489, 3)
Train used for fitting: (20888, 3)


In [10]:
vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    ngram_range=NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF
)

X_train = vectorizer.fit_transform(train_fit_df['text'])
X_val = vectorizer.transform(val_df['text'])
X_test = vectorizer.transform(test_df['text'])

print('X_train shape:', X_train.shape)
print('X_val shape  :', X_val.shape)
print('X_test shape :', X_test.shape)

model = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    random_state=RANDOM_STATE
)

model.fit(X_train, train_fit_df['label'])
print('\nModel training complete.')
print('Model intercept:', float(model.intercept_[0]))

X_train shape: (20888, 5000)
X_val shape  : (53489, 5000)
X_test shape : (53489, 5000)

Model training complete.
Model intercept: 1.9076226347142498


In [11]:
def find_best_threshold(y_true, scores, min_precision=0.90):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    rows = []
    for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
        f1 = 0.0 if (p + r) == 0 else (2 * p * r) / (p + r)
        rows.append((float(t), float(p), float(r), float(f1)))

    threshold_df = pd.DataFrame(rows, columns=['threshold', 'precision', 'recall', 'f1'])
    preferred = threshold_df[threshold_df['precision'] >= min_precision]

    if not preferred.empty:
        return preferred.sort_values(['f1', 'recall'], ascending=False).iloc[0], threshold_df
    return threshold_df.sort_values('f1', ascending=False).iloc[0], threshold_df


val_scores = model.predict_proba(X_val)[:, 1]
best_row, threshold_table = find_best_threshold(
    val_df['label'].values,
    val_scores,
    min_precision=MIN_PRECISION_FOR_THRESHOLD
)

PHISHING_THRESHOLD = round(float(best_row['threshold']), 4)
SUSPICIOUS_THRESHOLD = round(max(0.25, PHISHING_THRESHOLD - 0.20), 4)

print('Chosen phishing threshold:', PHISHING_THRESHOLD)
print('Chosen suspicious threshold:', SUSPICIOUS_THRESHOLD)
display(best_row.to_frame().T)

Chosen phishing threshold: 0.8889
Chosen suspicious threshold: 0.6889


,threshold,precision,recall,f1
52296,0.888939,0.900151,0.911179,0.905632


In [12]:
test_scores = model.predict_proba(X_test)[:, 1]
test_pred = (test_scores >= PHISHING_THRESHOLD).astype(int)

print('ROC-AUC:', round(roc_auc_score(test_df['label'], test_scores), 4))
print('\nClassification report:')
print(classification_report(test_df['label'], test_pred, target_names=['Legitimate', 'Phishing']))

cm = confusion_matrix(test_df['label'], test_pred)
cm_df = pd.DataFrame(
    cm,
    index=['Actual Legitimate', 'Actual Phishing'],
    columns=['Predicted Legitimate', 'Predicted Phishing']
)
display(cm_df)

ROC-AUC: 0.9996

Classification report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     52837
    Phishing       0.92      0.90      0.91       652

    accuracy                           1.00     53489
   macro avg       0.96      0.95      0.95     53489
weighted avg       1.00      1.00      1.00     53489



,Predicted Legitimate,Predicted Phishing
Actual Legitimate,52784,53
Actual Phishing,63,589


In [13]:
results = test_df[['raw_text', 'text', 'label']].copy()
results['score'] = test_scores
results['pred'] = test_pred

false_positives = results[(results['label'] == 0) & (results['pred'] == 1)].sort_values('score', ascending=False)
false_negatives = results[(results['label'] == 1) & (results['pred'] == 0)].sort_values('score', ascending=True)

print('False positives:', len(false_positives))
display(false_positives.head(10))

print('False negatives:', len(false_negatives))
display(false_negatives.head(10))

False positives: 53


,raw_text,text,label,score,pred
526844,http : / / hrweb . enron . com / benefits / formp . asp,http hrweb com benefits formp asp,0,0.993828,1
534824,registration confirmation from spinner . com thank you for joining spinner . com : the web ' s largest source of free streaming music just wanted to confirm your registration w...,registration confirmation from spinner com thank you for joining spinner com the web s largest source of free streaming music just wanted to confirm your registration with spin...,0,0.982505,1
520445,"BIG 12 OR BUST Â This weekend you can see the Nebraska Cornhuskers, Utah State, Kansas State and the University of Louisiana/Monroe - LIVE AND EXCLUSIVELY ON THE INTERNET! FOXS...",big or bust this weekend you can see the nebraska cornhuskers utah state kansas state and the university of louisiana monroe live and exclusively on the internet foxsports com ...,0,0.981263,1
532799,info hpuente @ epelectric . com 915 - 543 - 4333,info hpuente epelectric com,0,0.979573,1
531477,digitize your memories with compaq scanners tired of searching for your photos or important files ? quit looking . we have a solution . we  ' re offering a compaq tested and a...,digitize your memories with compaq scanners tired of searching for your photos or important files quit looking we have a solution we re offering a compaq tested and approved ag...,0,0.972806,1
528473,"> This is getting better and better. You know 3K is utter shite how? I'm not going to be your spokesman, Eugen. Figure it out yourself.over and out.",this is getting better and better you know 3k is utter shite how i m not going to be your spokesman eugen figure it out yourself over and out,0,0.968373,1
71908,"Message-ID: Date: Mon, 21 Jan 2002 13:16:43 -0800 (PST) From: technology_buy.com@enews.buy.com To: don.baughman@enron.com Subject: Great PC stuff at great prices! Mime-Version:...",mon jan pst great pc stuff at great prices text plain charset ansi_x3 7bit x buy com x baughman jr don x x b exmerge baughman jr don deleted items baughman d don baughman pst g...,0,0.967172,1
154232,"Message-ID: Date: Tue, 16 Jan 2001 04:42:00 -0800 (PST) From: vince.kaminski@enron.com To: paulbunion@prodigy.net Subject: Remove Mime-Version: 1.0 Content-Type: text/plain; ch...",tue jan pst remove text plain charset us ascii 7bit x vince j kaminski x x x b vincent_kaminski_jun2001_4 notes folders sent mail kaminski v vkamins nsf remove on am please res...,0,0.960889,1
208260,"Message-ID: Date: Wed, 16 Jan 2002 13:20:48 -0800 (PST) From: entertainment_buy.com@enews.buy.com To: mike.carson@enron.com Subject: DVDs as low as $9.49.... Mime-Version: 1.0 ...",wed jan pst dvds as low as text plain charset ansi_x3 7bit x buy com x carson mike x x b exmerge carson mike deleted items carson m mike carson pst dear michael now that you ow...,0,0.959212,1
72264,"Message-ID: Date: Fri, 21 Dec 2001 14:21:06 -0800 (PST) From: buy.com@enews.buy.com To: don.baughman@enron.com Subject: Be merry with After Holiday Blowout Savings! Mime-Versio...",fri dec pst be merry with after holiday blowout savings text plain charset ansi_x3 7bit x buy com x baughman jr don x x b edward_baughman_jan2002 baughman jr don inbox baughman...,0,0.958177,1


False negatives: 63


,raw_text,text,label,score,pred
525371,"wndows x , p home update minnesota , which can clinch a wild - card playoff spot with a loss by either carolina or st . louis this weekend , appeared on its way to retaking the...",wndows x p home update minnesota which can clinch a wild card playoff spot with a loss by either carolina or st louis this weekend appeared on its way to retaking the lead but ...,1,0.554206,0
530259,"exclusive positions in montanayoocwo sjk xl 20551 fancy image being downloaded pythagoras , ancient greek mathematician d 7 i 334 ndwsmkrsvpjlfygao 37405 e 5213 3124 357 0184 b...",exclusive positions in montanayoocwo sjk xl fancy image being downloaded pythagoras ancient greek mathematician d i ndwsmkrsvpjlfygao e bowy xnmnurvno ltewvsoih cagoyjqjlodhnj ...,1,0.598169,0
521397,"small cap stox can sizzle the contact center industry few people today know exactly what a contact center does . its predecessor , the call center , brings to mind large telema...",small cap stox can sizzle the contact center industry few people today know exactly what a contact center does its predecessor the call center brings to mind large telemarketin...,1,0.647464,0
528836,creativity and innovation newsletter - march 2005 vol 1 creativity andinnovation newsletter march 2005 seek a creativity driven future out - think your competition the topic of...,creativity and innovation newsletter march vol creativity andinnovation newsletter march seek a creativity driven future out think your competition the topic of innovation has ...,1,0.664099,0
528012,"schedualed meeting starts on april 30 th israel approved plans this week for a major offensive in gaza if the palestinian authority fails to stop the attacks , but israeli offi...",schedualed meeting starts on april th israel approved plans this week for a major offensive in gaza if the palestinian authority fails to stop the attacks but israeli officials...,1,0.668384,0
531157,"news alert - wall street research - otc spni investor alert ! if a group of investors would have the opportunity to have invest in the nba , nfl , nbl , nhl , and mls when the ...",news alert wall street research otc spni investor alert if a group of investors would have the opportunity to have invest in the nba nfl nbl nhl and mls when the leagues starte...,1,0.714197,0
519693,thief - proofing a car thief - proofing a car peripheral connections proves it can be done peripheral connection ( otcbb : pepo ) has made the world a safer place - at least if...,thief proofing a car thief proofing a car peripheral connections proves it can be done peripheral connection otcbb pepo has made the world a safer place at least if you are a c...,1,0.717645,0
534489,"spyglass maxine , . . . . nightdress first we would like to say thank you to al | of our avid readers ! we have had huge success over the last few months and have become one of...",spyglass maxine nightdress first we would like to say thank you to al of our avid readers we have had huge success over the last few months and have become one of the most wide...,1,0.735702,0
530405,"this works the latter allowed it to come within half a cable ' s length ; then , as if disdaining to dive , it took a little turn , and stopped a short distance off no msg two ...",this works the latter allowed it to come within half a cable s length then as if disdaining to dive it took a little turn and stopped a short distance off no msg two days passe...,1,0.736795,0
523633,"returned mail : see transcript for details the original message was received at tue , 19 jul 2005 07 : 06 : 09 - 0400 from root @ localhost - - - - - the following addresses ha...",returned mail see transcript for details the original message was received at tue jul from root localhost the following addresses had permanent fatal errors antique reason can ...,1,0.743162,0


In [14]:
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

top_phishing_idx = np.argsort(coefficients)[-20:][::-1]
top_safe_idx = np.argsort(coefficients)[:20]

top_phishing = pd.DataFrame({'feature': feature_names[top_phishing_idx], 'weight': coefficients[top_phishing_idx]})
top_safe = pd.DataFrame({'feature': feature_names[top_safe_idx], 'weight': coefficients[top_safe_idx]})

print('Top phishing features:')
display(top_phishing)

print('Top safe features:')
display(top_safe)

Top phishing features:


,feature,weight
0,http,5.142019
1,your,3.246040
2,here,3.158944
3,www,3.130251
4,money,3.082706
5,you,2.971700
6,our,2.865589
7,statements,2.473307
8,no,2.316347
9,remove,2.286363


Top safe features:


,feature,weight
0,pst,-6.720303
1,plain charset,-5.522757
2,text plain,-5.510952
3,plain,-5.439408
4,7bit,-5.319948
5,text,-5.274801
6,charset,-5.267678
7,ascii 7bit,-5.139482
8,charset us,-5.076758
9,us ascii,-5.071724


In [15]:
MODEL_PATH = OUTPUT_DIR / 'phishing_model.pkl'
VECTORIZER_PATH = OUTPUT_DIR / 'tfidf_vectorizer.pkl'
METADATA_PATH = OUTPUT_DIR / 'training_metadata.json'

joblib.dump(vectorizer, VECTORIZER_PATH)
joblib.dump(model, MODEL_PATH)

metadata = {
    'raw_dataset_path': str(RAW_DATASET_PATH),
    'rows_ready_for_training': int(len(df)),
    'train_rows': int(len(train_df)),
    'train_fit_rows': int(len(train_fit_df)),
    'validation_rows': int(len(val_df)),
    'test_rows': int(len(test_df)),
    'balance_train_split': BALANCE_TRAIN_SPLIT,
    'max_safe_to_phishing_ratio': MAX_SAFE_TO_PHISHING_RATIO,
    'phishing_threshold': PHISHING_THRESHOLD,
    'suspicious_threshold': SUSPICIOUS_THRESHOLD,
    'roc_auc': float(roc_auc_score(test_df['label'], test_scores))
}

METADATA_PATH.write_text(json.dumps(metadata, indent=2))

print('Saved:', VECTORIZER_PATH)
print('Saved:', MODEL_PATH)
print('Saved:', METADATA_PATH)

Saved: /content/drive/MyDrive/fyp/retrained_backend/tfidf_vectorizer.pkl
Saved: /content/drive/MyDrive/fyp/retrained_backend/phishing_model.pkl
Saved: /content/drive/MyDrive/fyp/retrained_backend/training_metadata.json


In [17]:
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

top_phishing_idx = np.argsort(coefficients)[-20:][::-1]
top_safe_idx = np.argsort(coefficients)[:20]

print("Top phishing features:")
for i in top_phishing_idx:
    print(feature_names[i], round(float(coefficients[i]), 4))

print("\nTop safe features:")
for i in top_safe_idx:
    print(feature_names[i], round(float(coefficients[i]), 4))


Top phishing features:
http 5.142
your 3.246
here 3.1589
www 3.1303
money 3.0827
you 2.9717
our 2.8656
statements 2.4733
no 2.3163
remove 2.2864
software 2.2027
http www 2.1114
viagra 2.0892
best 2.0551
over 1.9733
quality 1.8825
bank 1.8084
investment 1.7853
only 1.7818
hello 1.7523

Top safe features:
pst -6.7203
plain charset -5.5228
text plain -5.511
plain -5.4394
7bit -5.3199
text -5.2748
charset -5.2677
ascii 7bit -5.1395
charset us -5.0768
us ascii -5.0717
ascii -5.0716
notes -4.9874
privileged -4.4173
folders -4.3777
non privileged -4.3495
notes folders -4.3269
nsf -4.3256
ect -4.1509
pdt -4.1157
non -4.0925
